# Rotary Position Embeddings (RoPE) from scratch

Walkthrough for **ai-learn-26-rope-embeddings**.

RoPE rotates query/key pairs by position so attention scores depend on **relative** offset `(m - n)` -- the scheme behind LLaMA, RoFormer, and most modern LLMs.

## 1. Build frequencies and apply RoPE

For even dimension `d` and base `theta=10000`:

theta_i = 10000^(-2i/d), RoPE(x, m) rotates each pair by m*theta_i.

Compact form: `RoPE(x) = x * cos + rotate_half(x) * sin`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
from rope import rope_freqs, apply_rope_pair, apply_rope, check_relative_property, rope_cos_sin

freqs = rope_freqs(dim=32, base=10000.0)
print('first 4 theta_i:', freqs[:4])
print('last 4 theta_i:', freqs[-4:])

x = np.random.default_rng(0).normal(size=32)
x0 = apply_rope_pair(x, 0)
x5 = apply_rope_pair(x, 5)
print('pos0 ~= x?', np.allclose(x0, x))
print('||x|| preserved?', np.allclose(np.linalg.norm(x), np.linalg.norm(x5)))

## 2. Relative property

`<RoPE(q, m), RoPE(k, n)> = <RoPE(q, m-n), k>`. Shifting both positions by delta leaves the score unchanged.

In [ ]:
from rope import rope_dot, relative_rope_kernel

rng = np.random.default_rng(42)
q, k = rng.normal(size=32), rng.normal(size=32)
m, n = 10, 3
print('abs form:', rope_dot(q, k, m, n))
print('rel form:', relative_rope_kernel(q, k, m - n))
print('shifted :', rope_dot(q, k, m + 7, n + 7))
print(check_relative_property(seed=42))

## 3. Relative-shift invariance of attention logits

Compare tiny attention with no PE / absolute sinusoidal / RoPE when all positions shift by delta.

In [ ]:
from experiments import relative_shift_invariance

inv = relative_shift_invariance(seed=42)
for pe, s in inv['summary'].items():
    print(f"{pe:10s} mean_cosine={s['mean_cosine']} min={s['min_cosine']}")

## 4. Relative-offset retrieval

Query must attend to the key at relative offset `-4`. Content-matched distractors make pure content matching fail; absolute PE overfits slots; RoPE tracks the offset.

In [ ]:
from experiments import train_relative_task

for pe in ('none', 'absolute', 'rope'):
    r = train_relative_task(pe, seed=42)
    print(f"{pe:10s} acc={r['final_acc']} mass={r['final_mass']} logit_gap={r['final_logit_gap']}")

## 5. Full smoke

```bash
python run_smoke.py   # writes results/ (seed 42)
```

See `results/RESULTS.md` for tables and SVG plots.